# fio-cmp: DAOS (dfs) vs NVMe-oF (libaio)

Sequential read comparison across block sizes 4K / 1M / 4M.
Single job, iodepth=32. Metrics: bandwidth, IOPS, latency percentiles, CPU usage.

In [1]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

RESULTS_DIR = Path(".")

BS_ORDER = ["4k", "1m", "4m"]
COLORS   = {"daos": "#1f77b4", "nsf": "#ff7f0e"}


## Network File System (NFS) Result Analysis


### pvsync2 engine

1. bs = 4m
10 repeated runs of `pvsync2` ioengine on NSF (seq read, bs=4M, nj=1, iodepth=32) to assess run-to-run variance.

In [3]:
import statistics

pvsync2_files = sorted(RESULTS_DIR.glob("pvsync2_nsf_*.json"))
records = []
for fpath in pvsync2_files:
    with open(fpath) as f:
        d = json.load(f)
    r = d["jobs"][0]["read"]
    records.append(dict(
        file       = fpath.name,
        bw_GiBs    = r["bw_bytes"] / (1024**3),
        iops       = r["iops"],
        lat_mean_ms= r["lat_ns"]["mean"] / 1e6,
        clat_p50_ms= r["clat_ns"]["percentile"]["50.000000"] / 1e6,
    ))

pv_df = pd.DataFrame(records)

# Summary stats
summary = {}
for col in ["bw_GiBs", "iops", "lat_mean_ms", "clat_p50_ms"]:
    vals = pv_df[col].tolist()
    mu = statistics.mean(vals)
    sd = statistics.stdev(vals)
    summary[col] = dict(min=min(vals), max=max(vals), mean=mu, std=sd, cv_pct=sd/mu*100)

summary_df = pd.DataFrame(summary).T.round(3)
summary_df.index.name = "metric"
print("Per-run results:")
print(pv_df.to_string(index=False))
print("\nAggregate stats (n=10):")
print(summary_df)

Per-run results:
                                               file  bw_GiBs       iops  lat_mean_ms  clat_p50_ms
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810430.json 0.926836 237.269983     4.213711     4.014080
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810467.json 0.819096 209.688522     4.247546     4.046848
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810507.json 0.832879 213.217067     4.165243     3.948544
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810546.json 0.833755 213.441158     4.163269     3.981312
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810586.json 0.833682 213.422466     4.159203     3.948544
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810626.json 0.837176 214.317123     4.135810     3.948544
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810665.json 0.830916 212.714577     4.171020     3.981312
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810705.json 0.843706 215.988655     4.108622     3.915776
pvsync2_nsf_seq_read_bs4m_nj1_iod32_1781810744.json 0.832685 213.167332     4.169557     3.948544
pvs

### Consistency findings

| Metric | Min | Max | Mean | Std | CV |
|---|---|---|---|---|---|
| BW (GiB/s) | 0.819 | 0.926 | 0.840 | 0.031 | 3.7% |
| IOPS | 209.7 | 237.3 | 215.3 | 7.9 | 3.7% |
| lat\_mean (ms) | 4.11 | 4.25 | 4.18 | 0.04 | 1.0% |
| clat\_p50 (ms) | 3.92 | 4.05 | 3.97 | 0.04 | 1.0% |

**Run 1 is an outlier:** the first run (`1781810430`) shows ~10% higher BW/IOPS (0.926 GiB/s, 237 IOPS) compared to the 9 subsequent runs (0.819–0.843 GiB/s). This is consistent with a warm-kernel-buffer or page-cache effect on the first run.

**Runs 2–10 are consistent:** CV ≈ 1–2% on bandwidth and <1% on latency — normal fio measurement noise. These runs are suitable as a stable baseline for pvsync2 on NVMe-oF.